# is_scam: xlm-roberta-base, обучение на Kaggle GPU T4

Бинарная детекция скама на уровне диалога. Обучение на `train` (silver-метки LLM),
валидация на `val` (silver), **тест только по ручной золотой разметке**.

**Что положить в Kaggle Dataset** (один датасет, оба файла в корне):

| файл в репозитории | назначение |
|---|---|
| `data/corpus/corpus_v1.jsonl` | корпус 2000 диалогов с полем `split` (train 1392 / val 205 / test 403) |
| `data/gold/gold_test_v1.csv` | ручная разметка теста, 403 строки, колонка `dialog_id` |

Датасет подключается в Kaggle как `/kaggle/input/<slug>/`. Путь задаётся в ячейке
«Конфигурация» — при необходимости поправить `INPUT_DIR`.

**Настройки Kaggle:** Accelerator = GPU T4 x1, Internet = On (нужен для загрузки
весов `xlm-roberta-base` с HuggingFace).

Гиперпараметры и сиды — те же, что в локальном `scripts/train_binary.py`.
Ожидаемый потолок качества ~0.86–0.90: согласие silver-меток с gold по `is_scam` = 0.859.

## Конфигурация

In [ ]:
import os, sys, json, csv, glob, pathlib, time

# Файлы ищутся РЕКУРСИВНО: Kaggle монтирует датасет по-разному
# (/kaggle/input/<slug>/ либо /kaggle/input/datasets/<...>/), на глубину не закладываемся.
print("дерево /kaggle/input:")
for root, dirs, files in os.walk("/kaggle/input"):
    depth = root.rstrip("/").count("/") - 2
    print("  " * depth + os.path.basename(root) + "/")
    for f in files:
        print("  " * (depth + 1) + f)

def find_one(name, required=True):
    hits = sorted(glob.glob(f"/kaggle/input/**/{name}", recursive=True))
    if not hits and not required:
        print(f"(необязательный файл {name} не найден — пропускаем)")
        return None
    assert hits, (f"не найден {name} под /kaggle/input — проверь, что датасет "
                  f"подключён к ноутбуку (Add Input) и содержит этот файл")
    if len(hits) > 1:
        print(f"⚠ найдено несколько {name}, беру первый:", hits)
    return hits[0]

CORPUS = find_one("corpus_v1.jsonl")
GOLD   = find_one("gold_test_v1.csv")
GOLD_DEV = find_one("gold_dev_split.json", required=False)   # отбор чекпоинта по gold
INPUT_DIR = os.path.dirname(CORPUS)
OUT    = pathlib.Path("/kaggle/working/binary_xlmr")
PLOTS  = OUT / "plots"
PLOTS.mkdir(parents=True, exist_ok=True)
TB_ROOT = pathlib.Path("/kaggle/working/tb")

MODEL_NAME  = "xlm-roberta-base"
SEED        = 20260721
LR          = 2e-5
WEIGHT_DECAY= 0.01
WARMUP_RATIO= 0.2   # 0.1 давало сырой grad_norm до 191 при max_grad_norm=1.0
EPOCHS      = 5
MAX_LENGTH  = 512   # доля диалогов длиннее 512 токенов = 5.10% (посчитано на корпусе)
BATCH_SIZE  = 16

print()
print("корпус:", CORPUS)
print("gold:  ", GOLD)

## Окружение и устройство

In [ ]:
import torch, transformers, sklearn
print("python      ", sys.version.split()[0])
print("torch       ", torch.__version__)
print("transformers", transformers.__version__)
print("sklearn     ", sklearn.__version__)

if torch.cuda.is_available():
    DEVICE, USE_FP16 = "cuda", True
    print("устройство: cuda —", torch.cuda.get_device_name(0))
else:
    DEVICE, USE_FP16 = "cpu", False
    print("устройство: cpu (GPU не подключён — проверь Accelerator в настройках ноутбука)")
print("fp16 =", USE_FP16)

## Данные

Сплиты берутся из поля `split` и НЕ переразбиваются. Текст реплик — `[индекс|говорящий] текст`, по одной реплике на строку.

In [ ]:
def read_corpus(path):
    out = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                out.append(json.loads(line))
    return out

def build_text(rec):
    """Реплики → текст: одна реплика на строку, формат "[индекс|говорящий] текст"."""
    return "\n".join(f"[{t['i']}|{t['speaker']}] {t['text']}" for t in rec["turns"])

def read_gold_csv(path):
    """Золотая разметка → {dialog_id: is_scam}. Файл только читается."""
    out = {}
    with open(path, encoding="utf-8-sig", newline="") as f:
        for r in csv.DictReader(f):
            did = (r.get("dialog_id") or "").strip()
            if did:
                out[did] = int(r["is_scam"])
    return out

records = read_corpus(CORPUS)
gold    = read_gold_csv(GOLD)

train_recs = [r for r in records if r.get("split") == "train"]
val_recs   = [r for r in records if r.get("split") == "val"]

# ТЕСТ: исходную silver-метку is_scam выбрасываем, метки берём только из gold
test_split = [r for r in records if r.get("split") == "test"]
test_recs, unmatched = [], []
for r in test_split:
    r2 = {k: v for k, v in r.items() if k != "is_scam"}
    if r2["id"] in gold:
        r2["gold_is_scam"] = gold[r2["id"]]
        test_recs.append(r2)
    else:
        unmatched.append(r2["id"])
gold_only = sorted(set(gold) - {r["id"] for r in test_split})

print(f"джойн gold ↔ test-сплит: смэтчилось {len(test_recs)} из {len(gold)} строк gold")
if len(test_recs) < len(gold):
    print("id из gold, которых нет в test-сплите (до 10):", gold_only[:10])
    print("id из test-сплита без gold-метки (до 10):", unmatched[:10])
    raise SystemExit("остановка: джойн неполный")

# gold-dev: часть золотого теста уходит на ОТБОР чекпоинта (человеческие метки).
# Отбор по silver-val бессмыслен: recall там насыщен в 1.0 с первой эпохи.
dev_recs = []
if GOLD_DEV:
    assign = json.load(open(GOLD_DEV, encoding="utf-8"))["assignment"]
    dev_recs  = [r for r in test_recs if assign.get(r["id"]) == "dev"]
    test_recs = [r for r in test_recs if assign.get(r["id"]) != "dev"]
    print(f"gold-dev включён: отбор чекпоинта по {len(dev_recs)}, финальный тест {len(test_recs)}")
else:
    print("gold-dev ВЫКЛЮЧЕН — отбор чекпоинта пойдёт по silver-val (сигнал насыщен)")

print(f"train {len(train_recs)} (silver) / val {len(val_recs)} (silver) / "
      f"gold-dev {len(dev_recs)} (gold) / test {len(test_recs)} (gold)")
print("\nпример текста:\n" + build_text(train_recs[0])[:400] + " …")

## Распределение длин и гистограмма

In [ ]:
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained(MODEL_NAME)
lens = np.array([len(tok(build_text(r), add_special_tokens=True)["input_ids"]) for r in records])
frac = float((lens > 512).mean())
print(f"медиана {np.median(lens):.0f}  90-й перцентиль {np.percentile(lens,90):.0f}  "
      f"максимум {lens.max()}  доля>512 {frac:.2%}")
print(f"max_length={MAX_LENGTH}, batch_size={BATCH_SIZE}")

fig, ax = plt.subplots(figsize=(9,5))
ax.hist(lens, bins=60, color="#4C72B0", edgecolor="white", linewidth=0.4)
ax.axvline(MAX_LENGTH, color="#C44E52", linestyle="--", linewidth=2, label=f"выбранный max_length = {MAX_LENGTH}")
ax.axvline(np.median(lens), color="#55A868", linestyle=":", linewidth=1.6, label=f"медиана = {np.median(lens):.0f}")
ax.axvline(np.percentile(lens,90), color="#8172B2", linestyle=":", linewidth=1.6, label=f"90-й перцентиль = {np.percentile(lens,90):.0f}")
ax.set_xlabel("длина диалога, токенов xlm-roberta-base")
ax.set_ylabel("число диалогов")
ax.set_title(f"Распределение длин диалогов корпуса (n={len(lens)}), доля длиннее 512 токенов — {frac:.1%}")
ax.legend(); ax.grid(axis="y", alpha=0.25); fig.tight_layout()
fig.savefig(PLOTS/"token_lengths.png", dpi=150); plt.show()

## Обучение

In [ ]:
from transformers import (AutoModelForSequenceClassification, EarlyStoppingCallback,
                          Trainer, TrainingArguments, set_seed)
from sklearn.metrics import (accuracy_score, cohen_kappa_score,
                             precision_recall_fscore_support)

set_seed(SEED)

class DlgDataset(torch.utils.data.Dataset):
    def __init__(self, recs, label_key):
        self.enc = tok([build_text(r) for r in recs], truncation=True,
                       max_length=MAX_LENGTH, padding="max_length")
        self.labels = [int(r[label_key]) for r in recs]
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, i):
        item = {k: torch.tensor(v[i]) for k, v in self.enc.items()}
        item["labels"] = torch.tensor(self.labels[i])
        return item

ds_train = DlgDataset(train_recs, "is_scam")
ds_val   = DlgDataset(val_recs,   "is_scam")
ds_test  = DlgDataset(test_recs,  "gold_is_scam")
ds_dev   = DlgDataset(dev_recs,   "gold_is_scam") if dev_recs else None

if ds_dev is not None:
    eval_sets, best_metric = {"silver_val": ds_val, "gold_dev": ds_dev}, "eval_gold_dev_f1"
else:
    eval_sets, best_metric = ds_val, "f1"

RUN_NAME = f"{MODEL_NAME.split('/')[-1]}_seed{SEED}_e{EPOCHS}"
TB_DIR = TB_ROOT / RUN_NAME
TB_DIR.mkdir(parents=True, exist_ok=True)
# transformers >= 5.1 игнорирует logging_dir; ставим и переменную, и аргумент
os.environ["TENSORBOARD_LOGGING_DIR"] = str(TB_DIR)
print("TensorBoard:", TB_DIR, "| отбор чекпоинта по:", best_metric)

def metrics_from(y_true, y_pred):
    p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, average="binary",
                                                 pos_label=1, zero_division=0)
    return {"accuracy": float(accuracy_score(y_true, y_pred)),
            "precision": float(p), "recall": float(r), "f1": float(f),
            "cohen_kappa": float(cohen_kappa_score(y_true, y_pred))}

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    return metrics_from(labels, np.argmax(logits, axis=-1))

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

targs = TrainingArguments(
    output_dir=str(OUT/"checkpoints"),
    learning_rate=LR, weight_decay=WEIGHT_DECAY, warmup_ratio=WARMUP_RATIO,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE, per_device_eval_batch_size=BATCH_SIZE,
    seed=SEED, data_seed=SEED, fp16=USE_FP16,
    eval_strategy="epoch", save_strategy="epoch",
    load_best_model_at_end=True, metric_for_best_model=best_metric, greater_is_better=True,
    save_total_limit=2, logging_steps=10,
    report_to="tensorboard", logging_dir=str(TB_DIR),
)

trainer = Trainer(model=model, args=targs,
                  train_dataset=ds_train, eval_dataset=eval_sets,
                  processing_class=tok, compute_metrics=compute_metrics,
                  callbacks=[EarlyStoppingCallback(early_stopping_patience=2)])

t0 = time.time()
trainer.train()
print(f"\nобучение заняло {(time.time()-t0)/60:.1f} мин")

history = trainer.state.log_history
(OUT/"history.json").write_text(json.dumps(history, ensure_ascii=False, indent=2), encoding="utf-8")
best_ckpt = trainer.state.best_model_checkpoint
print("лучший чекпоинт:", best_ckpt, "| val f1 =", trainer.state.best_metric)

## Ход обучения по эпохам

In [ ]:
_pfx = "eval_gold_dev_" if dev_recs else "eval_"
ev = [h for h in history if f"{_pfx}loss" in h]
for h in ev:   # унифицируем имена, чтобы графики не зависели от режима
    for k in ("loss","accuracy","precision","recall","f1","cohen_kappa"):
        if f"{_pfx}{k}" in h: h[f"eval_{k}"] = h[f"{_pfx}{k}"]
print(f"{'эпоха':>6} {'train loss':>11} {'val loss':>9} {'val acc':>8} {'val F1':>8} {'val κ':>8}")
tr = [h for h in history if "loss" in h and "eval_loss" not in h]
for h in ev:
    ep = h["epoch"]
    near = [x["loss"] for x in tr if x["epoch"] <= ep]
    tl = sum(near[-5:]) / len(near[-5:]) if near else float("nan")
    print(f"{ep:6.0f} {tl:11.4f} {h['eval_loss']:9.4f} {h['eval_accuracy']:8.4f} "
          f"{h['eval_f1']:8.4f} {h['eval_cohen_kappa']:8.4f}")

best_ep = max(ev, key=lambda h: h["eval_f1"])["epoch"] if ev else None
print(f"\nлучшая эпоха по val F1: {best_ep:g}")
print(f"эпох отработано: {len(ev)} из {EPOCHS} " +
      ("→ сработала ранняя остановка (patience=2)" if len(ev) < EPOCHS else "→ ранняя остановка не срабатывала"))

## Тест на золотой разметке

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

pred_out = trainer.predict(ds_test)
logits = pred_out.predictions
probs  = torch.softmax(torch.tensor(logits), dim=-1).numpy()[:, 1]
y_pred = logits.argmax(axis=-1)
y_true = np.array([r["gold_is_scam"] for r in test_recs])

m  = metrics_from(y_true, y_pred)
cm = confusion_matrix(y_true, y_pred, labels=[0,1])

print("="*60); print(f"ТЕСТ на ручной золотой разметке (n={len(y_true)})"); print("="*60)
for k in ("accuracy","precision","recall","f1","cohen_kappa"):
    print(f"  {k:12s} {m[k]:.4f}")
print("\nматрица ошибок (строки — gold, столбцы — предсказание):")
print("            pred=0  pred=1")
print(f"  gold=0    {cm[0,0]:6d}  {cm[0,1]:6d}")
print(f"  gold=1    {cm[1,0]:6d}  {cm[1,1]:6d}")
print("\n" + classification_report(y_true, y_pred, labels=[0,1],
                                   target_names=["benign","scam"], digits=4, zero_division=0))

per_lang = {}
langs = sorted({r.get("lang") for r in test_recs if r.get("lang")})
if langs:
    print("качество по языкам (поле lang):")
    for lg in langs:
        idx = [i for i,r in enumerate(test_recs) if r.get("lang")==lg]
        mm = metrics_from(y_true[idx], y_pred[idx]); mm["n"] = len(idx)
        per_lang[lg] = mm
        print(f"  {lg}: n={mm['n']:3d}  acc={mm['accuracy']:.4f}  f1={mm['f1']:.4f}  κ={mm['cohen_kappa']:.4f}")
else:
    print("⚠ поля языка в данных нет — разбивка по языкам не считается")

with (OUT/"test_predictions.csv").open("w", encoding="utf-8", newline="") as f:
    w = csv.writer(f)
    w.writerow(["dialog_id","lang","gold_is_scam","pred_is_scam","prob_scam"])
    for r, yt, yp, pr in zip(test_recs, y_true, y_pred, probs):
        w.writerow([r["id"], r.get("lang",""), int(yt), int(yp), f"{pr:.6f}"])

(OUT/"test_results.json").write_text(json.dumps({
    "device": DEVICE, "model": MODEL_NAME, "seed": SEED,
    "max_length": MAX_LENGTH, "batch_size": BATCH_SIZE, "epochs_requested": EPOCHS,
    "best_model_checkpoint": best_ckpt, "best_val_f1": trainer.state.best_metric,
    "join_report": {"n_test_split": len(test_split), "n_gold_rows": len(gold), "n_matched": len(test_recs)},
    "test_metrics": m, "confusion_matrix_gold_x_pred": cm.tolist(), "per_language": per_lang,
}, ensure_ascii=False, indent=2), encoding="utf-8")

from torch.utils.tensorboard import SummaryWriter
w = SummaryWriter(log_dir=str(TB_DIR))
for k, v in m.items(): w.add_scalar(f"gold_test/{k}", v, 0)
for k, v in (pred_out.metrics or {}).items():
    if isinstance(v, (int, float)): w.add_scalar(f"gold_test/{k.replace('test_','')}", v, 0)
for lg, mm in per_lang.items():
    w.add_scalar(f"gold_test_f1/{lg}", mm["f1"], 0)
    w.add_scalar(f"gold_test_recall/{lg}", mm["recall"], 0)
w.add_scalar("gold_test/n", len(y_true), 0); w.flush(); w.close()
print("итог по gold записан в TensorBoard:", TB_DIR)

trainer.save_model(str(OUT/"best")); tok.save_pretrained(str(OUT/"best"))
print("\nартефакты записаны в", OUT)

## Графики

Те же шесть, что строит локальный `scripts/make_train_plots.py`. Все точки — из
`history.json` и `test_predictions.csv` этого прогона, ничего не сглаживается.

In [ ]:
DPI = 150
tr_pts = [h for h in history if "loss" in h and "eval_loss" not in h]
lr_pts = [h for h in history if "learning_rate" in h]
steps_per_epoch = tr_pts[-1]["step"]/tr_pts[-1]["epoch"] if tr_pts and tr_pts[-1].get("epoch") else None
created, skipped = [], []

def vline(ax, epoch, as_step=False):
    if epoch is None: return
    x = epoch*steps_per_epoch if (as_step and steps_per_epoch) else epoch
    ax.axvline(x, color="#C44E52", linestyle="--", linewidth=1.8,
               label=f"лучший чекпоинт (эпоха {epoch:g})")

# a) learning_curve
if tr_pts and ev:
    fig, ax = plt.subplots(figsize=(9,5))
    ax.plot([h["step"] for h in tr_pts], [h["loss"] for h in tr_pts],
            color="#4C72B0", linewidth=1.2, label="train loss (по шагам)")
    ax.plot([h["step"] for h in ev], [h["eval_loss"] for h in ev],
            color="#DD8452", marker="o", linewidth=2, label="val loss (по эпохам)")
    vline(ax, best_ep, as_step=True)
    ax.set_xlabel("шаг обучения"); ax.set_ylabel("функция потерь")
    ax.set_title("Кривая обучения: потери на train (по шагам) и на val (по эпохам)")
    ax.legend(); ax.grid(alpha=0.25); fig.tight_layout()
    fig.savefig(PLOTS/"learning_curve.png", dpi=DPI); plt.show(); created.append("learning_curve.png")
else:
    skipped.append("learning_curve.png — нет точек 'loss'/'eval_loss' в истории")

# b) val_metrics
keys = [("eval_accuracy","accuracy","#4C72B0"),("eval_f1","F1","#DD8452"),("eval_cohen_kappa","Cohen κ","#55A868")]
have = [(k,l,c) for k,l,c in keys if any(k in h for h in ev)]
if have:
    fig, ax = plt.subplots(figsize=(9,5))
    for k,l,c in have:
        pts = [(h["epoch"], h[k]) for h in ev if k in h]
        ax.plot([x for x,_ in pts], [y for _,y in pts], marker="o", color=c, linewidth=2, label=l)
    vline(ax, best_ep)
    ax.set_xlabel("эпоха"); ax.set_ylabel("значение метрики")
    ax.set_title("Метрики на валидации по эпохам (accuracy, F1, Cohen κ)")
    ax.legend(); ax.grid(alpha=0.25); fig.tight_layout()
    fig.savefig(PLOTS/"val_metrics.png", dpi=DPI); plt.show(); created.append("val_metrics.png")
else:
    skipped.append("val_metrics.png — нет eval_accuracy/eval_f1/eval_cohen_kappa в истории")

# c) confusion_matrix
fig, ax = plt.subplots(figsize=(6,5.4))
im = ax.imshow(cm, cmap="Blues")
for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm[i,j]), ha="center", va="center", fontsize=20,
                color="white" if cm[i,j] > cm.max()/2 else "#222")
ax.set_xticks([0,1], ["benign (0)","scam (1)"]); ax.set_yticks([0,1], ["benign (0)","scam (1)"])
ax.set_xlabel("предсказание"); ax.set_ylabel("gold")
ax.set_title(f"Матрица ошибок на золотом тесте (n={len(y_true)})")
fig.colorbar(im, ax=ax, shrink=0.8, label="число диалогов"); fig.tight_layout()
fig.savefig(PLOTS/"confusion_matrix.png", dpi=DPI); plt.show(); created.append("confusion_matrix.png")

# d) per_language
if per_lang:
    langs_ = sorted(per_lang); f1s = [per_lang[l]["f1"] for l in langs_]; ns = [per_lang[l]["n"] for l in langs_]
    fig, ax = plt.subplots(figsize=(8,5))
    bars = ax.bar(langs_, f1s, color="#4C72B0", width=0.55)
    for b,n,v in zip(bars, ns, f1s):
        ax.text(b.get_x()+b.get_width()/2, v+0.015, f"n={n}", ha="center", fontsize=11)
    ax.set_ylim(0, min(1.05, max(f1s)+0.12))
    ax.set_xlabel("язык"); ax.set_ylabel("F1 (класс scam)")
    ax.set_title("F1 по языкам на золотом тесте (над столбцом — число примеров)")
    ax.grid(axis="y", alpha=0.25); fig.tight_layout()
    fig.savefig(PLOTS/"per_language.png", dpi=DPI); plt.show(); created.append("per_language.png")
else:
    skipped.append("per_language.png — в данных нет поля 'lang'")

# e) confidence_hist
correct = (y_true == y_pred)
fig, ax = plt.subplots(figsize=(9,5))
bins = np.linspace(0,1,31)
ax.hist(probs[correct],  bins=bins, alpha=0.55, color="#55A868", label=f"верно классифицированы (n={int(correct.sum())})")
ax.hist(probs[~correct], bins=bins, alpha=0.55, color="#C44E52", label=f"ошибки (n={int((~correct).sum())})")
ax.set_xlabel("предсказанная вероятность класса scam"); ax.set_ylabel("число диалогов")
ax.set_title("Уверенность модели на золотом тесте: верные предсказания против ошибок")
ax.legend(); ax.grid(axis="y", alpha=0.25); fig.tight_layout()
fig.savefig(PLOTS/"confidence_hist.png", dpi=DPI); plt.show(); created.append("confidence_hist.png")

# f) lr_schedule
if lr_pts:
    fig, ax = plt.subplots(figsize=(9,5))
    ax.plot([h["step"] for h in lr_pts], [h["learning_rate"] for h in lr_pts], color="#8172B2", linewidth=2)
    vline(ax, best_ep, as_step=True)
    ax.set_xlabel("шаг обучения"); ax.set_ylabel("learning rate")
    ax.set_title("Фактический learning rate по шагам (из истории обучения)")
    ax.legend(); ax.grid(alpha=0.25); fig.tight_layout()
    fig.savefig(PLOTS/"lr_schedule.png", dpi=DPI); plt.show(); created.append("lr_schedule.png")
else:
    skipped.append("lr_schedule.png — нет поля 'learning_rate' в истории")

print("создано:")
for f in created: print("  ", PLOTS/f)
if skipped:
    print("\nне построено:")
    for s in skipped: print("  ⚠", s)

## Что забрать обратно в репозиторий

Из `/kaggle/working/binary_xlmr/` скачать и положить в `runs/binary_xlmr/`:
`history.json`, `test_predictions.csv`, `test_results.json`, `plots/*.png`, `best/`.

После этого локальный `python scripts/make_train_plots.py` перестроит те же графики
из тех же артефактов — числа совпадут.